# Full benchmark run — dev (60) + locked eval (240)

Run `00_colab_smoke.ipynb` successfully first.

**Disconnect protocol:** checkpoints live on Drive with per-sample (per-pass for
leave-one-out) granularity. If the runtime dies: Reconnect → Runtime → Run all.
Completed samples are skipped; at most the in-flight unit is lost. A changed config
refuses to resume (new run required) — that is intentional.

**Time budget (estimates, T4 FP16):** dev60 ≈ 1–2 h; eval240 ≈ 4.5–8 h — a free-tier
T4 session will likely NOT finish eval240 in one sitting; multi-session resume is the
designed path (or one Colab Pro A100 session ≈ 1–2 h).

**Lock discipline:** the eval split is locked. It is run once per benchmark version;
justify any re-run in the run manifest notes.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU runtime — Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0), "| BF16:", torch.cuda.is_bf16_supported())

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
BUNDLE_ZIP = '/content/drive/MyDrive/reab/reab_bundle.zip'
DRIVE_RESULTS = '/content/drive/MyDrive/reab/results'
assert os.path.exists(BUNDLE_ZIP), f'upload the bundle to {BUNDLE_ZIP} first'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.environ['RAG_EVIDENCE_RESULTS_RAW'] = f'{DRIVE_RESULTS}/raw'
os.environ['RAG_EVIDENCE_RESULTS_DERIVED'] = f'{DRIVE_RESULTS}/derived'
os.environ['HF_HOME'] = '/content/hf_cache'

In [ ]:
!rm -rf /content/reab && mkdir -p /content/reab
!unzip -q "$BUNDLE_ZIP" -d /content/reab
%cd /content/reab
!pip install -q "transformers==5.14.1" "tokenizers==0.22.2" "accelerate>=1.0" "bitsandbytes>=0.49" "datasets>=3.0" "sentence-transformers>=5.0"
!pip install -q -e ".[ml,gpu]"
!mkdir -p "$DRIVE_RESULTS" && cp -rn results/raw "$DRIVE_RESULTS/" 2>/dev/null; cp -rn results/derived "$DRIVE_RESULTS/" 2>/dev/null; true

## Section 1 — dev split (60 questions)

In [ ]:
!python -m rag_evidence.cli status --config configs/dev.yaml

In [ ]:
# dense retrieval for dev/eval runs here (GPU-fast); bm25 came with the bundle
!python -m rag_evidence.cli retrieve --method dense      --config configs/dev.yaml --resume
!python -m rag_evidence.cli retrieve --method hybrid_rrf --config configs/dev.yaml --resume

In [ ]:
!python -m rag_evidence.cli generate --config configs/dev.yaml --resume

In [ ]:
%%bash
for m in citations embedding leave_one_out control_random control_retrieval control_lexical control_length control_shuffled; do
  python -m rag_evidence.cli attribute --method $m --config configs/dev.yaml --resume
done

In [ ]:
!python -m rag_evidence.cli evaluate --config configs/dev.yaml
!python -m rag_evidence.cli report   --config configs/dev.yaml
!python -m rag_evidence.cli export   --config configs/dev.yaml

## Section 2 — LOCKED eval split (240 questions)

Longest section. Safe to run across several sessions — rerun from the status cell
after any disconnect.

In [ ]:
!python -m rag_evidence.cli status --config configs/full.yaml

In [ ]:
!python -m rag_evidence.cli retrieve --method dense      --config configs/full.yaml --resume
!python -m rag_evidence.cli retrieve --method hybrid_rrf --config configs/full.yaml --resume

In [ ]:
!python -m rag_evidence.cli generate --config configs/full.yaml --resume

In [ ]:
%%bash
for m in citations embedding leave_one_out control_random control_retrieval control_lexical control_length control_shuffled; do
  python -m rag_evidence.cli attribute --method $m --config configs/full.yaml --resume
done

In [ ]:
!python -m rag_evidence.cli evaluate --config configs/full.yaml
!python -m rag_evidence.cli report   --config configs/full.yaml
!python -m rag_evidence.cli export   --config configs/full.yaml
import glob
from google.colab import files
zips = sorted(glob.glob('results/export/*.zip'))
print('export zips:', zips)
if zips:
    files.download(zips[-1])